In [1]:
# Cell 1 (NEW - Simplified for Colab)
!git clone https://github.com/Kuzay3t/ViT-NAS-Compression.git
%cd ViT-NAS-Compression

# Install core dependencies only (skip setup.py)
!pip install -q torch torchvision transformers timm numpy pyyaml omegaconf tqdm tensorboard wandb onnx onnxruntime psutil

print("✅ Dependencies installed!")

Cloning into 'ViT-NAS-Compression'...
remote: Enumerating objects: 64, done.
remote: Counting objects: 100% (64/64), done.
remote: Compressing objects: 100% (60/60), done.
remote: Total 64 (delta 22), reused 5 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (64/64), 35.53 KiB | 4.44 MiB/s, done.
Resolving deltas: 100% (22/22), done.
/kaggle/working/ViT-NAS-Compression
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 89.8 MB/s eta 0:00:00
✅ Dependencies installed!


In [2]:
# Cell 2
from src.utils.device_info import DeviceInfo
from src.search_space.search_space import SearchSpace

print("✅ All imports successful!")

✅ All imports successful!


In [3]:
# Cell 3
DeviceInfo.print_system_info()


SYSTEM INFORMATION

--- CPU & Memory ---
platform...................... Linux
platform_release.............. 6.6.113+
processor..................... x86_64
cpu_count..................... 4
cpu_freq_ghz.................. 2.00
ram_gb........................ 31.35
available_ram_gb.............. 30.01

--- PyTorch ---
pytorch_version............... 2.10.0+cu128
cuda_available................ True
cuda_version.................. 12.8
cuda_device_count............. 2
cuda_devices.................. ['Tesla T4', 'Tesla T4']
cudnn_version................. 91002
cudnn_enabled................. True

--- GPU ---
GPU 0: Tesla T4
  Memory: 14913 MB
GPU 1: Tesla T4
  Memory: 14913 MB



In [4]:
# Cell 4
search_space = SearchSpace("config/search_space.yaml")
search_space.print_search_space_info()

[2026-04-27 20:00:38] [src.search_space.search_space] [INFO] SearchSpace initialized from config/search_space.yaml

SEARCH SPACE INFORMATION

--- ARCHITECTURE ---
depth......................... 4 choices: [6, 9, 12, 16]
embed_dim..................... 4 choices: [192, 384, 576, 768]
num_heads..................... 4 choices: [3, 6, 9, 12]
mlp_ratio..................... 4 choices: [2.0, 3.0, 4.0, 5.0]
patch_size.................... 3 choices: [8, 16, 32]

--- COMPRESSION ---
Pruning methods............... 4 methods

--- HARDWARE ---
Target devices................ 6 devices



In [5]:
# Cell 5 (Debug version)
from src.search_space.search_space import SearchSpace

search_space = SearchSpace("config/search_space.yaml")

# Generate a config
config = search_space.random_sample()
is_valid, errors = search_space.validate_config(config)

print("="*70)
print("CONFIG VALIDATION DEBUG")
print("="*70)

print(f"\n✓ Config generated successfully")
print(f"\nArchitecture:")
print(f"  - depth: {config.architecture.depth}")
print(f"  - embed_dim: {config.architecture.embed_dim}")
print(f"  - num_heads: {config.architecture.num_heads}")

print(f"\nCompression:")
print(f"  - pruning enabled: {config.compression.pruning.enabled}")
print(f"  - quantization enabled: {config.compression.quantization.enabled}")
print(f"  - pruning ratios: {config.compression.pruning.layer_wise_ratios[:3]}...")
print(f"  - quant bits: {config.compression.quantization.layer_wise_bits[:3]}...")

print(f"\nValidation Result: {is_valid}")

if not is_valid:
    print(f"\n❌ ERRORS FOUND:")
    for i, error in enumerate(errors, 1):
        print(f"  {i}. {error}")
else:
    print(f"\n✅ Config is VALID!")

print("="*70)

[2026-04-27 20:00:38] [src.search_space.search_space] [INFO] SearchSpace initialized from config/search_space.yaml
CONFIG VALIDATION DEBUG

✓ Config generated successfully

Architecture:
  - depth: 16
  - embed_dim: 384
  - num_heads: 6

Compression:
  - pruning enabled: False
  - quantization enabled: False
  - pruning ratios: [0.1012880727281339, 0.1373590022711043, 0.0884032276893395]...
  - quant bits: [8, 16, 8]...

Validation Result: True

✅ Config is VALID!


In [6]:
# ═══════════════════════════════════════════════════════════════════════════════
# Cell 6  →  REPLACE
# FIX: Added structured pruning masks + quantisation-aware forward pass inside
#      each TransformerBlock so the NAS actually applies compression config.
#      Added exit_threshold as a property so the supernet is query-adaptive.
# ═══════════════════════════════════════════════════════════════════════════════

import os
os.makedirs('src/supernet', exist_ok=True)

with open('src/supernet/__init__.py', 'w') as f:
    f.write('# Supernet modules\n')

vit_code = '''import torch
import torch.nn as nn
import torch.nn.functional as F
import time
from typing import Optional, Tuple


# ── Quantisation helper ───────────────────────────────────────────────────────

def fake_quantise(x: torch.Tensor, bits: int) -> torch.Tensor:
    """Straight-through quantisation-aware forward pass.
    Rounds weights/activations to `bits`-bit grid during forward,
    passes full gradient on backward (straight-through estimator).
    bits=8  → standard INT8 deployment
    bits=4  → aggressive INT4 deployment
    bits=16 → full precision (no-op)
    """
    if bits >= 16:
        return x
    n_levels = 2 ** bits
    x_min, x_max = x.min(), x.max()
    scale = (x_max - x_min) / (n_levels - 1) + 1e-8
    x_q = torch.round((x - x_min) / scale) * scale + x_min
    # straight-through: forward uses quantised, backward uses real
    return x + (x_q - x).detach()


# ── Building blocks ───────────────────────────────────────────────────────────

class PatchEmbedding(nn.Module):
    def __init__(self, img_size=32, patch_size=4, in_channels=3, embed_dim=384):
        super().__init__()
        self.num_patches = (img_size // patch_size) ** 2
        self.proj = nn.Conv2d(in_channels, embed_dim,
                              kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        return self.proj(x).flatten(2).transpose(1, 2)


class Attention(nn.Module):
    def __init__(self, dim, num_heads=6):
        super().__init__()
        self.num_heads = num_heads
        self.scale = (dim // num_heads) ** -0.5
        self.qkv  = nn.Linear(dim, dim * 3, bias=True)
        self.proj = nn.Linear(dim, dim)

    def forward(self, x, pruning_mask: Optional[torch.Tensor] = None,
                quant_bits: int = 16):
        B, N, C = x.shape
        # Optionally quantise QKV weights
        qkv_w = fake_quantise(self.qkv.weight, quant_bits)
        qkv   = F.linear(x, qkv_w, self.qkv.bias)
        qkv   = qkv.reshape(B, N, 3, self.num_heads, C // self.num_heads).permute(2, 0, 3, 1, 4)
        q, k, v = qkv.unbind(0)

        attn = (q @ k.transpose(-2, -1) * self.scale).softmax(dim=-1)

        # Head pruning: zero out heads whose mask entry == 0
        if pruning_mask is not None:
            # pruning_mask shape: (num_heads,) — 1 = keep, 0 = prune
            mask = pruning_mask.to(attn.device).view(1, -1, 1, 1)
            attn = attn * mask

        out = (attn @ v).transpose(1, 2).reshape(B, N, C)
        proj_w = fake_quantise(self.proj.weight, quant_bits)
        return F.linear(out, proj_w, self.proj.bias)


class MLP(nn.Module):
    def __init__(self, dim, mlp_ratio=4.0):
        super().__init__()
        hidden = int(dim * mlp_ratio)
        self.fc1 = nn.Linear(dim, hidden)
        self.fc2 = nn.Linear(hidden, dim)
        self.act = nn.GELU()

    def forward(self, x, pruning_ratio: float = 0.0, quant_bits: int = 16):
        w1 = fake_quantise(self.fc1.weight, quant_bits)
        h  = self.act(F.linear(x, w1, self.fc1.bias))

        # Neuron pruning: zero out the bottom `pruning_ratio` fraction of neurons
        if pruning_ratio > 0.0:
            hidden = h.shape[-1]
            n_prune = int(hidden * pruning_ratio)
            if n_prune > 0:
                # prune neurons with smallest mean activation magnitude
                importance = h.abs().mean(dim=(0, 1))  # (hidden,)
                _, prune_idx = importance.topk(n_prune, largest=False)
                mask = torch.ones(hidden, device=h.device)
                mask[prune_idx] = 0.0
                h = h * mask

        w2 = fake_quantise(self.fc2.weight, quant_bits)
        return F.linear(h, w2, self.fc2.bias)


class EarlyExitGate(nn.Module):
    """Lightweight confidence head placed after selected blocks."""
    def __init__(self, dim: int, num_classes: int = 10):
        super().__init__()
        self.norm = nn.LayerNorm(dim)
        self.head = nn.Linear(dim, num_classes)

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, float]:
        """Returns (logits, confidence) where confidence = max softmax prob."""
        cls = self.norm(x[:, 0])
        logits = self.head(cls)
        confidence = logits.softmax(dim=-1).max(dim=-1).values.mean().item()
        return logits, confidence


class TransformerBlock(nn.Module):
    def __init__(self, dim, num_heads, mlp_ratio=4.0, drop=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn  = Attention(dim, num_heads)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp   = MLP(dim, mlp_ratio)
        self.drop  = nn.Dropout(drop)
        self.num_heads = num_heads

    def forward(self, x,
                pruning_ratio: float = 0.0,
                quant_bits: int = 16):
        """
        Args:
            x             : (B, N, dim)
            pruning_ratio : fraction of MLP neurons to prune (0 = off)
            quant_bits    : bit-width for QAT (16 = full precision)
        """
        # Build head-pruning mask from pruning_ratio
        # We prune the bottom fraction of heads (sorted by index for determinism)
        n_prune_heads = int(self.num_heads * pruning_ratio)
        if n_prune_heads > 0:
            head_mask = torch.ones(self.num_heads)
            head_mask[:n_prune_heads] = 0.0          # prune first n heads
        else:
            head_mask = None

        x = x + self.drop(self.attn(self.norm1(x),
                                     pruning_mask=head_mask,
                                     quant_bits=quant_bits))
        x = x + self.drop(self.mlp(self.norm2(x),
                                    pruning_ratio=pruning_ratio,
                                    quant_bits=quant_bits))
        return x


# ── Supernet ──────────────────────────────────────────────────────────────────

# Exit gates are inserted after these block indices (0-indexed)
EXIT_GATE_LAYERS = [2, 5, 8, 11]  # after blocks 3, 6, 9, 12 (0-indexed)


class ViTSupernet(nn.Module):
    """
    ViT supernet for CIFAR-10 with:
      - Weight-sharing across depth subnets (sandwich rule training)
      - Per-layer pruning ratio applied during forward
      - Per-layer quantisation bit-width (straight-through QAT)
      - Early-exit gates at layers [3,6,9,12] for query-adaptive inference

    All dimensions are fixed for CIFAR-10:
      img_size=32, patch_size=4, embed_dim=384, num_heads=6, max_depth=12
    """
    def __init__(self, img_size=32, patch_size=4, in_channels=3,
                 num_classes=10, max_depth=12, embed_dim=384, num_heads=6):
        super().__init__()
        self.max_depth = max_depth
        self.embed_dim = embed_dim
        self.num_heads = num_heads

        self.patch_embed = PatchEmbedding(img_size, patch_size, in_channels, embed_dim)
        num_patches = self.patch_embed.num_patches

        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches + 1, embed_dim))
        self.pos_drop  = nn.Dropout(p=0.1)

        self.blocks = nn.ModuleList([
            TransformerBlock(embed_dim, num_heads, mlp_ratio=4.0)
            for _ in range(max_depth)
        ])

        # Early-exit gates (placed at EXIT_GATE_LAYERS)
        self.exit_gates = nn.ModuleDict({
            str(i): EarlyExitGate(embed_dim, num_classes)
            for i in EXIT_GATE_LAYERS if i < max_depth
        })

        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, num_classes)

        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.trunc_normal_(m.weight, std=0.02)
            if m.bias is not None:
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.LayerNorm):
            nn.init.constant_(m.bias, 0)
            nn.init.constant_(m.weight, 1.0)

    def forward(self, x,
                depth: Optional[int] = None,
                pruning_ratios: Optional[list] = None,
                quant_bits: Optional[list] = None,
                exit_threshold: Optional[float] = None):
        """
        Args:
            x               : (B, C, H, W) — must be 32×32 for CIFAR-10
            depth           : number of active blocks (None = max_depth)
            pruning_ratios  : list of per-layer float pruning ratios, length=depth
            quant_bits      : list of per-layer int bit-widths, length=depth
            exit_threshold  : if set, enables early-exit at confidence > threshold
                              Returns (logits, metadata) when threshold is given.

        Returns:
            logits only when exit_threshold is None (training mode)
            (logits, metadata) when exit_threshold is set (inference mode)
        """
        depth = depth or self.max_depth
        B = x.shape[0]

        x = self.patch_embed(x)
        cls = self.cls_token.expand(B, -1, -1)
        x   = self.pos_drop(torch.cat([cls, x], dim=1) + self.pos_embed)

        for i in range(depth):
            pr  = pruning_ratios[i] if pruning_ratios else 0.0
            qb  = quant_bits[i]     if quant_bits     else 16
            x   = self.blocks[i](x, pruning_ratio=pr, quant_bits=qb)

            # Early exit check (inference only, not during sandwich training)
            if exit_threshold is not None and str(i) in self.exit_gates:
                logits, confidence = self.exit_gates[str(i)](x)
                if confidence > exit_threshold:
                    return logits, {
                        "early_exited": True,
                        "exit_layer": i + 1,
                        "layers_used": i + 1,
                        "confidence": confidence,
                    }

        logits = self.head(self.norm(x[:, 0]))

        if exit_threshold is not None:
            return logits, {
                "early_exited": False,
                "exit_layer": None,
                "layers_used": depth,
                "confidence": 1.0,
            }
        return logits

    # ── Utility methods ───────────────────────────────────────────────────────

    def measure_latency(self, depth, device, n_runs=50, warmup=10):
        """Real inference latency for a depth setting (ms)."""
        self.eval()
        dummy = torch.randn(1, 3, 32, 32).to(device)
        with torch.no_grad():
            for _ in range(warmup):
                self(dummy, depth=depth)
            if device.type == "cuda":
                torch.cuda.synchronize()
            t0 = time.perf_counter()
            for _ in range(n_runs):
                self(dummy, depth=depth)
            if device.type == "cuda":
                torch.cuda.synchronize()
        return (time.perf_counter() - t0) / n_runs * 1000

    def count_active_params(self, depth):
        """Parameters actually used at a given depth (millions)."""
        fixed = (
            sum(p.numel() for p in self.patch_embed.parameters()) +
            self.cls_token.numel() + self.pos_embed.numel() +
            sum(p.numel() for p in self.norm.parameters()) +
            sum(p.numel() for p in self.head.parameters())
        )
        per_block = sum(p.numel() for p in self.blocks[0].parameters())
        return (fixed + depth * per_block) / 1e6
'''

with open('src/supernet/vit_supernet.py', 'w') as f:
    f.write(vit_code)

print("Created src/supernet/vit_supernet.py")
print("  img_size=32  patch_size=4  embed_dim=384  num_heads=6  max_depth=12")
print("  + per-layer pruning  + per-layer QAT  + early-exit gates")



Created src/supernet/vit_supernet.py
  img_size=32  patch_size=4  embed_dim=384  num_heads=6  max_depth=12
  + per-layer pruning  + per-layer QAT  + early-exit gates


In [7]:
# ═══════════════════════════════════════════════════════════════════════════════
# Cell 7  →  REPLACE
# FIX: Was testing with img_size=224 / num_classes=1000 (ImageNet dims).
#      Now uses CIFAR-10 dims (32×32 / 10 classes) consistent with the supernet.
#      Removed `config=config` kwarg — supernet.forward() takes `depth`, not config.
# ═══════════════════════════════════════════════════════════════════════════════

import torch
from src.supernet.vit_supernet import ViTSupernet
from src.search_space.search_space import SearchSpace

print("=" * 70)
print("ViT SUPERNET TEST  (CIFAR-10 dims)")
print("=" * 70)

model = ViTSupernet(img_size=32, patch_size=4, num_classes=10,
                    max_depth=16, embed_dim=384, num_heads=6)
print(f"\n✓ ViT Supernet created")

num_params = sum(p.numel() for p in model.parameters())
print(f"✓ Total parameters: {num_params:,}")

# CIFAR-10 images are 32×32
x = torch.randn(2, 3, 32, 32)
print(f"✓ Input shape: {x.shape}")

# --- Test 1: basic forward (no compression, no early exit) ---
output = model(x)
print(f"✓ Output shape (full depth, no compression): {output.shape}")
assert output.shape == (2, 10), f"Expected (2,10), got {output.shape}"

# --- Test 2: depth subnet ---
search_space = SearchSpace("config/search_space.yaml")
config = search_space.random_sample()
output_depth = model(x, depth=config.architecture.depth)
print(f"✓ Output with depth={config.architecture.depth}: {output_depth.shape}")

# --- Test 3: pruning + quantisation forward ---
depth = config.architecture.depth
pr = config.compression.pruning.layer_wise_ratios[:depth]
qb = config.compression.quantization.layer_wise_bits[:depth]
output_compressed = model(x, depth=depth, pruning_ratios=pr, quant_bits=qb)
print(f"✓ Output with pruning+quant (depth={depth}): {output_compressed.shape}")

# --- Test 4: early-exit inference ---
logits, meta = model(x, depth=16, exit_threshold=0.05)  # low threshold → likely exits early
print(f"✓ Early-exit: exited={meta['early_exited']}  "
      f"layers_used={meta['layers_used']}  confidence={meta['confidence']:.4f}")

print(f"\n✅ ViT Supernet Test PASSED!")
print("=" * 70)


ViT SUPERNET TEST  (CIFAR-10 dims)

✓ ViT Supernet created
✓ Total parameters: 28,458,674
✓ Input shape: torch.Size([2, 3, 32, 32])
✓ Output shape (full depth, no compression): torch.Size([2, 10])
[2026-04-27 20:00:39] [src.search_space.search_space] [INFO] SearchSpace initialized from config/search_space.yaml
✓ Output with depth=6: torch.Size([2, 10])
✓ Output with pruning+quant (depth=6): torch.Size([2, 10])
✓ Early-exit: exited=True  layers_used=3  confidence=0.1753

✅ ViT Supernet Test PASSED!


In [8]:
##############################################################################
# NEW CELL 7A — CIFAR-10 Data Loading
# ADD this as a brand-new cell right after Cell 7 (the supernet smoke test).
# proxy_train_loader and proxy_val_loader stay in memory for NAS evaluation.
##############################################################################

# ── Paste everything below into new Cell 7A ───────────────────────────────

import torch
import torchvision
import torchvision.transforms as T
from torch.utils.data import DataLoader, Subset
import numpy as np

mean = (0.4914, 0.4822, 0.4465)
std  = (0.2470, 0.2435, 0.2616)

train_transform = T.Compose([
    T.RandomCrop(32, padding=4),
    T.RandomHorizontalFlip(),
    T.ToTensor(),
    T.Normalize(mean, std),
])
val_transform = T.Compose([T.ToTensor(), T.Normalize(mean, std)])

train_full = torchvision.datasets.CIFAR10('./data', train=True,  download=True, transform=train_transform)
val_full   = torchvision.datasets.CIFAR10('./data', train=False, download=True, transform=val_transform)

rng = np.random.default_rng(42)

# Proxy subset — used per-config during NAS (fast)
proxy_train_idx = rng.choice(len(train_full), 5_000,  replace=False)
proxy_val_idx   = rng.choice(len(val_full),   1_000,  replace=False)

proxy_train_loader = DataLoader(Subset(train_full, proxy_train_idx),
                                batch_size=128, shuffle=True,  num_workers=2, pin_memory=True)
proxy_val_loader   = DataLoader(Subset(val_full,   proxy_val_idx),
                                batch_size=256, shuffle=False, num_workers=2, pin_memory=True)

# Supernet pre-training subset (larger, one-time cost)
supernet_train_idx = rng.choice(len(train_full), 20_000, replace=False)
supernet_train_loader = DataLoader(Subset(train_full, supernet_train_idx),
                                   batch_size=128, shuffle=True, num_workers=2, pin_memory=True)

print(f"proxy_train_loader  : 5,000 samples  ({len(proxy_train_loader)} batches)")
print(f"proxy_val_loader    : 1,000 samples  ({len(proxy_val_loader)} batches)")
print(f"supernet_train_loader: 20,000 samples ({len(supernet_train_loader)} batches)")


100%|██████████| 170M/170M [00:12<00:00, 14.2MB/s]


proxy_train_loader  : 5,000 samples  (40 batches)
proxy_val_loader    : 1,000 samples  (4 batches)
supernet_train_loader: 20,000 samples (157 batches)


In [9]:
# ═══════════════════════════════════════════════════════════════════════════════
# Cell 7B  — Supernet pre-training (sandwich rule + gate loss, single pass)
# ═══════════════════════════════════════════════════════════════════════════════

import torch
import torch.nn as nn
import random, time
from src.supernet.vit_supernet import ViTSupernet, EXIT_GATE_LAYERS

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on: {device}")
if device.type != "cuda":
    print("TIP: switch to T4 GPU in Colab for ~10x faster training.")

supernet = ViTSupernet(
    img_size=32, patch_size=4, num_classes=10,
    max_depth=16, embed_dim=384, num_heads=6
).to(device)
print(f"Supernet params: {sum(p.numel() for p in supernet.parameters())/1e6:.1f}M")

SUPERNET_EPOCHS = 5
MIN_DEPTH, MAX_DEPTH = 4, 12
GATE_LOSS_WEIGHT = 0.3

optimiser = torch.optim.AdamW(supernet.parameters(), lr=3e-4, weight_decay=0.05)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimiser, T_max=SUPERNET_EPOCHS)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

def _forward_with_gate_losses(supernet, imgs, labels, depth, criterion):
    """
    Single forward pass through `depth` blocks.
    Computes the main classification loss at the final block AND
    a gate loss at every exit gate encountered along the way.
    Returns total loss (main + weighted gate losses).
    """
    B = imgs.shape[0]
    x = supernet.patch_embed(imgs)
    cls = supernet.cls_token.expand(B, -1, -1)
    x = supernet.pos_drop(torch.cat([cls, x], dim=1) + supernet.pos_embed)

    total_loss = torch.tensor(0.0, device=imgs.device)

    for i in range(depth):
        x = supernet.blocks[i](x)
        # Gate loss: if this block has an exit gate, train it with a
        # down-weighted cross-entropy so it learns meaningful confidence scores
        if str(i) in supernet.exit_gates:
            gate_logits, _ = supernet.exit_gates[str(i)](x)
            total_loss = total_loss + GATE_LOSS_WEIGHT * criterion(gate_logits, labels)

    # Main classification loss at the end of this depth
    logits = supernet.head(supernet.norm(x[:, 0]))
    total_loss = total_loss + criterion(logits, labels)
    return total_loss

for epoch in range(SUPERNET_EPOCHS):
    supernet.train()
    running_loss, correct, total = 0.0, 0, 0
    t0 = time.time()

    for imgs, labels in supernet_train_loader:   # defined in Cell 7A
        imgs, labels = imgs.to(device), labels.to(device)
        optimiser.zero_grad()

        # Sandwich rule: max depth + one random middle + min depth
        depths = [MAX_DEPTH,
                  random.randint(MIN_DEPTH + 1, MAX_DEPTH - 1),
                  MIN_DEPTH]

        # One pass per depth (no redundant re-forward)
        total_loss = sum(
            _forward_with_gate_losses(supernet, imgs, labels, d, criterion)
            for d in depths
        ) / len(depths)

        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(supernet.parameters(), 1.0)
        optimiser.step()

        with torch.no_grad():
            preds = supernet(imgs, depth=MAX_DEPTH).argmax(1)
            correct += (preds == labels).sum().item()
            total   += labels.size(0)
            running_loss += total_loss.item()

    scheduler.step()
    acc = correct / total * 100
    avg_loss = running_loss / len(supernet_train_loader)
    print(f"Epoch {epoch+1}/{SUPERNET_EPOCHS}  "
          f"loss={avg_loss:.4f}  acc@depth12={acc:.1f}%  "
          f"({time.time()-t0:.0f}s)")

torch.save(supernet.state_dict(), 'supernet_checkpoint.pt')
print("\nCheckpoint saved → supernet_checkpoint.pt")

Training on: cuda
Supernet params: 28.5M
Epoch 1/5  loss=3.4764  acc@depth12=28.6%  (143s)
Epoch 2/5  loss=3.1905  acc@depth12=36.9%  (161s)
Epoch 3/5  loss=3.0033  acc@depth12=43.2%  (167s)
Epoch 4/5  loss=2.8020  acc@depth12=48.6%  (169s)
Epoch 5/5  loss=2.6925  acc@depth12=51.6%  (170s)

Checkpoint saved → supernet_checkpoint.pt


In [10]:
# ═══════════════════════════════════════════════════════════════════════════════
# Cell 9  — NAS Controller (corrected)
# Fixes:
#   1. mutation() now resizes pruning_ratios and quant_bits when depth changes
#   2. mutation() deep-copies config so it never mutates the original in-place
#   3. _effective_model_size() fixed byte-unit double-count
#   4. _proxy_train_and_eval() has safety clamp on list lengths
# ═══════════════════════════════════════════════════════════════════════════════

import os
os.makedirs('src/nas', exist_ok=True)

with open('src/nas/__init__.py', 'w') as f:
    f.write('# NAS modules\n')

nas_code = '''import random
import copy
import time
import torch
import torch.nn as nn
from typing import List, Dict, Any, Optional

EXIT_THRESHOLD_CHOICES = [0.3, 0.5, 0.7, 0.9]


class EvolutionaryNAS:
    """
    Evolutionary NAS that jointly searches:
      - Architecture depth
      - Per-layer pruning ratios (applied in forward pass)
      - Per-layer quantisation bit-widths (QAT in forward pass)
      - Early-exit threshold (query-adaptive computation)
    """

    def __init__(self, search_space, supernet, device,
                 proxy_train_loader, proxy_val_loader):
        self.search_space        = search_space
        self.supernet            = supernet
        self.device              = device
        self.proxy_train_loader  = proxy_train_loader
        self.proxy_val_loader    = proxy_val_loader
        self.fitness_history     = []
        self.pareto_front        = []
        print(f"EvolutionaryNAS initialised on {device}")

    # ── Real evaluation ───────────────────────────────────────────────────────

    def evaluate_config(self, config) -> Dict[str, float]:
        depth          = config.architecture.depth
        pruning_ratios = config.compression.pruning.layer_wise_ratios[:depth]
        quant_bits     = config.compression.quantization.layer_wise_bits[:depth]
        exit_threshold = getattr(config, "exit_threshold", 0.5)

        if not config.compression.pruning.enabled:
            pruning_ratios = [0.0] * depth
        if not config.compression.quantization.enabled:
            quant_bits = [16] * depth

        # Safety clamp: ensure lists are exactly length=depth
        pruning_ratios = _pad_or_clip(pruning_ratios, depth, 0.0)
        quant_bits     = _pad_or_clip(quant_bits,     depth, 16)

        accuracy   = self._proxy_train_and_eval(
            depth, pruning_ratios, quant_bits, exit_threshold)
        latency    = self._measure_latency(depth, exit_threshold)
        model_size = self._effective_model_size(depth, pruning_ratios, quant_bits)

        return {
            "accuracy":   accuracy,
            "latency":    latency,
            "model_size": model_size,
        }

    def _proxy_train_and_eval(self, depth, pruning_ratios, quant_bits,
                               exit_threshold) -> float:
        """Clone supernet, fine-tune head for 1 epoch, return val accuracy."""
        # Safety clamp (belt and braces)
        pruning_ratios = _pad_or_clip(pruning_ratios, depth, 0.0)
        quant_bits     = _pad_or_clip(quant_bits,     depth, 16)

        model = copy.deepcopy(self.supernet).to(self.device)
        model.train()

        params    = list(model.norm.parameters()) + list(model.head.parameters())
        optimiser = torch.optim.Adam(params, lr=1e-3)
        criterion = nn.CrossEntropyLoss()

        for imgs, labels in self.proxy_train_loader:
            imgs, labels = imgs.to(self.device), labels.to(self.device)
            optimiser.zero_grad()
            logits = model(imgs, depth=depth,
                           pruning_ratios=pruning_ratios,
                           quant_bits=quant_bits)
            if isinstance(logits, tuple):
                logits = logits[0]
            criterion(logits, labels).backward()
            optimiser.step()

        model.eval()
        correct = total = 0
        with torch.no_grad():
            for imgs, labels in self.proxy_val_loader:
                imgs, labels = imgs.to(self.device), labels.to(self.device)
                out = model(imgs, depth=depth,
                            pruning_ratios=pruning_ratios,
                            quant_bits=quant_bits,
                            exit_threshold=exit_threshold)
                logits = out[0] if isinstance(out, tuple) else out
                correct += (logits.argmax(1) == labels).sum().item()
                total   += labels.size(0)

        del model
        if self.device.type == "cuda":
            torch.cuda.empty_cache()

        return correct / total

    def _measure_latency(self, depth, exit_threshold, n_runs=50, warmup=10) -> float:
        self.supernet.eval()
        dummy = torch.randn(1, 3, 32, 32).to(self.device)
        with torch.no_grad():
            for _ in range(warmup):
                self.supernet(dummy, depth=depth, exit_threshold=exit_threshold)
            if self.device.type == "cuda":
                torch.cuda.synchronize()
            t0 = time.perf_counter()
            for _ in range(n_runs):
                self.supernet(dummy, depth=depth, exit_threshold=exit_threshold)
            if self.device.type == "cuda":
                torch.cuda.synchronize()
        return (time.perf_counter() - t0) / n_runs * 1000

    def _effective_model_size(self, depth, pruning_ratios, quant_bits) -> float:
        """
        Model size in MB accounting for pruning and quantisation.
        FIX: fixed params use 4 bytes each (float32).
             block params use qb/8 bytes each (quantised).
             No double-multiplication.
        """
        fixed_params = (
            sum(p.numel() for p in self.supernet.patch_embed.parameters()) +
            self.supernet.cls_token.numel() + self.supernet.pos_embed.numel() +
            sum(p.numel() for p in self.supernet.norm.parameters()) +
            sum(p.numel() for p in self.supernet.head.parameters())
        )
        per_block_base = sum(p.numel() for p in self.supernet.blocks[0].parameters())

        total_bytes = fixed_params * 4  # fixed always float32
        for i in range(depth):
            pr = pruning_ratios[i] if i < len(pruning_ratios) else 0.0
            qb = quant_bits[i]     if i < len(quant_bits)     else 16
            effective_params = per_block_base * (1.0 - pr * 0.5)
            bytes_per_param  = qb / 8.0
            total_bytes += effective_params * bytes_per_param

        return total_bytes / (1024 * 1024)

    # ── Fitness ───────────────────────────────────────────────────────────────

    def compute_fitness(self, metrics: Dict[str, float],
                        target_latency: float = 10.0) -> float:
        return (metrics["accuracy"]
                - 0.005 * max(0.0, metrics["latency"] - target_latency)
                - 0.002 * metrics["model_size"])

    # ── Population helpers ────────────────────────────────────────────────────

    def _sample_valid(self):
        config = self.search_space.random_sample()
        config.exit_threshold = random.choice(EXIT_THRESHOLD_CHOICES)
        valid, _ = self.search_space.validate_config(config)
        while not valid:
            config = self.search_space.random_sample()
            config.exit_threshold = random.choice(EXIT_THRESHOLD_CHOICES)
            valid, _ = self.search_space.validate_config(config)
        return config

    def random_population(self, pop_size: int) -> List[Dict]:
        return [{"config": self._sample_valid(),
                 "accuracy": 0.0, "latency": 0.0,
                 "model_size": 0.0, "fitness": 0.0}
                for _ in range(pop_size)]

    def evaluate_population(self, population: List[Dict]) -> List[Dict]:
        for i, ind in enumerate(population):
            d  = ind["config"].architecture.depth
            et = getattr(ind["config"], "exit_threshold", 0.5)
            print(f"  [{i+1}/{len(population)}] depth={d} exit_thr={et:.1f}", end="  ")
            t0 = time.time()
            m  = self.evaluate_config(ind["config"])
            ind.update(m)
            ind["fitness"] = self.compute_fitness(m)
            print(f"acc={m['accuracy']*100:.1f}%  lat={m['latency']:.1f}ms  "
                  f"size={m['model_size']:.1f}MB  ({time.time()-t0:.0f}s)")
        return population

    def selection(self, population: List[Dict], k: int = 3) -> List[Dict]:
        return [max(random.sample(population, k), key=lambda x: x["fitness"])
                for _ in range(len(population))]

    def crossover(self, p1, p2):
        child = self._sample_valid()
        child.architecture.depth = random.choice(
            [p1.architecture.depth, p2.architecture.depth])
        child.compression.pruning.enabled = random.choice(
            [p1.compression.pruning.enabled, p2.compression.pruning.enabled])
        child.compression.quantization.enabled = random.choice(
            [p1.compression.quantization.enabled, p2.compression.quantization.enabled])
        child.exit_threshold = random.choice(
            [getattr(p1, "exit_threshold", 0.5),
             getattr(p2, "exit_threshold", 0.5)])

        # Ensure compression lists match the child depth
        new_depth = child.architecture.depth
        child.compression.pruning.layer_wise_ratios = [
            random.uniform(0.0, 0.5) for _ in range(new_depth)
        ]
        child.compression.quantization.layer_wise_bits = [
            random.choice([4, 8, 16]) for _ in range(new_depth)
        ]

        valid, _ = self.search_space.validate_config(child)
        return child if valid else self._sample_valid()

    def mutation(self, config):
        """
        FIX 1: deep-copy so the original population member is never modified.
        FIX 2: after changing depth, resize pruning_ratios and quant_bits
                to exactly match the new depth — previously they stayed at the
                old length, causing IndexError in the forward pass.
        """
        config = copy.deepcopy(config)

        choice = random.choice(["depth", "pruning", "quant", "exit_threshold"])
        if choice == "depth":
            config.architecture.depth = random.choice([6, 9, 12, 16])
        elif choice == "pruning":
            config.compression.pruning.enabled = not config.compression.pruning.enabled
        elif choice == "quant":
            config.compression.quantization.enabled = not config.compression.quantization.enabled
        else:
            config.exit_threshold = random.choice(EXIT_THRESHOLD_CHOICES)

        # Always realign list lengths to current depth (fixes the IndexError)
        new_depth = config.architecture.depth
        config.compression.pruning.layer_wise_ratios = [
            random.uniform(0.0, 0.5) for _ in range(new_depth)
        ]
        config.compression.quantization.layer_wise_bits = [
            random.choice([4, 8, 16]) for _ in range(new_depth)
        ]

        return config

    # ── Pareto front ──────────────────────────────────────────────────────────

    def _compute_pareto_front(self, population: List[Dict]) -> List[Dict]:
        pareto = []
        for ind in population:
            dominated = any(
                (other["accuracy"]   >= ind["accuracy"]   and
                 other["latency"]    <= ind["latency"]    and
                 other["model_size"] <= ind["model_size"] and
                 (other["accuracy"]   > ind["accuracy"]   or
                  other["latency"]    < ind["latency"]    or
                  other["model_size"] < ind["model_size"]))
                for other in population if other is not ind
            )
            if not dominated:
                pareto.append(ind)
        return pareto

    # ── Main search loop ──────────────────────────────────────────────────────

    def search(self, num_generations: int = 5, pop_size: int = 8,
               mutation_rate: float = 0.3) -> Dict[str, Any]:
        total_evals = num_generations * pop_size
        print(f"Unified NAS  |  {num_generations} gen × {pop_size} pop = {total_evals} evals")
        print(f"Searching: depth + pruning_ratios + quant_bits + exit_threshold")
        print()

        population = self.random_population(pop_size)

        for gen in range(num_generations):
            print(f"Generation {gen+1}/{num_generations}")
            population = self.evaluate_population(population)

            best = max(population, key=lambda x: x["fitness"])
            self.fitness_history.append(best["fitness"])
            self.pareto_front = self._compute_pareto_front(population)

            print(f"  Best → acc={best['accuracy']*100:.1f}%  "
                  f"lat={best['latency']:.1f}ms  "
                  f"size={best['model_size']:.1f}MB  "
                  f"fitness={best['fitness']:.4f}  "
                  f"exit_thr={getattr(best['config'], 'exit_threshold', 0.5):.1f}  "
                  f"pareto={len(self.pareto_front)}")
            print()

            selected   = self.selection(population)
            population = []
            for _ in range(pop_size):
                p1, p2 = random.choice(selected), random.choice(selected)
                child  = self.crossover(p1["config"], p2["config"])
                if random.random() < mutation_rate:
                    child = self.mutation(child)
                valid, _ = self.search_space.validate_config(child)
                if not valid:
                    child = self._sample_valid()
                population.append({"config": child, "accuracy": 0.0,
                                   "latency": 0.0, "model_size": 0.0, "fitness": 0.0})

        best_overall = max(self.pareto_front, key=lambda x: x["fitness"])
        return {
            "best_config"    : best_overall["config"],
            "best_fitness"   : best_overall["fitness"],
            "best_accuracy"  : best_overall["accuracy"],
            "best_latency"   : best_overall["latency"],
            "best_size"      : best_overall["model_size"],
            "pareto_front"   : self.pareto_front,
            "fitness_history": self.fitness_history,
        }


def _pad_or_clip(lst, length, fill_value):
    """Ensure list is exactly `length` items — clip if too long, pad if too short."""
    lst = list(lst) if lst else []
    if len(lst) >= length:
        return lst[:length]
    return lst + [fill_value] * (length - len(lst))
'''

with open('src/nas/controller.py', 'w') as f:
    f.write(nas_code)

print("Created src/nas/controller.py")
print("  Joint search: depth + pruning + quant_bits + exit_threshold")
print("  Compression applied in forward pass (not just sampled)")

Created src/nas/controller.py
  Joint search: depth + pruning + quant_bits + exit_threshold
  Compression applied in forward pass (not just sampled)


In [11]:
# ═══════════════════════════════════════════════════════════════════════════════
# Cell 10  →  REPLACE
# FIX: Loads CIFAR-10 supernet (32×32, 10 classes) not ImageNet dims.
#      Removed target_latency / target_model_size kwargs that caused TypeError.
# ═══════════════════════════════════════════════════════════════════════════════

import torch
from src.supernet.vit_supernet import ViTSupernet
from src.search_space.search_space import SearchSpace
from src.nas.controller import EvolutionaryNAS

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type != "cuda":
    print("WARNING: No GPU found. Each eval ~3 min on CPU.")
    print("         Runtime > Change runtime type > T4 GPU")

supernet = ViTSupernet(
    img_size=32, patch_size=4, num_classes=10,
    max_depth=16, embed_dim=384, num_heads=6
).to(device)
supernet.load_state_dict(torch.load("supernet_checkpoint.pt", map_location=device))
supernet.eval()
print("Loaded supernet_checkpoint.pt")

search_space = SearchSpace("config/search_space.yaml")

# proxy_train_loader and proxy_val_loader come from Cell 7A
nas = EvolutionaryNAS(
    search_space       = search_space,
    supernet           = supernet,
    device             = device,
    proxy_train_loader = proxy_train_loader,
    proxy_val_loader   = proxy_val_loader,
)

# Quick smoke-test: num_generations=2, pop_size=4  (~5 min)
# Full run:         num_generations=5, pop_size=8  (~25 min)
results = nas.search(num_generations=5, pop_size=8, mutation_rate=0.3)

print("=" * 60)
print("NAS SEARCH COMPLETE")
print("=" * 60)
print(f"Best depth          : {results['best_config'].architecture.depth}")
print(f"Best exit threshold : {getattr(results['best_config'], 'exit_threshold', 0.5)}")
print(f"Best accuracy       : {results['best_accuracy']*100:.2f}%")
print(f"Best latency        : {results['best_latency']:.2f} ms")
print(f"Best size           : {results['best_size']:.2f} MB")
print(f"Fitness score       : {results['best_fitness']:.4f}")
print()
print(f"Pareto front ({len(results['pareto_front'])} configs):")
print(f"  {'Depth':>6}  {'ExitThr':>8}  {'Acc':>7}  {'Lat(ms)':>9}  {'Size(MB)':>9}")
for ind in sorted(results['pareto_front'], key=lambda x: -x['accuracy']):
    d  = ind['config'].architecture.depth
    et = getattr(ind['config'], 'exit_threshold', 0.5)
    print(f"  {d:>6}  {et:>8.1f}  {ind['accuracy']*100:>6.1f}%  "
          f"{ind['latency']:>8.1f}  {ind['model_size']:>8.1f}")


Device: cuda
Loaded supernet_checkpoint.pt
[2026-04-27 20:14:33] [src.search_space.search_space] [INFO] SearchSpace initialized from config/search_space.yaml
EvolutionaryNAS initialised on cuda
Unified NAS  |  5 gen × 8 pop = 40 evals
Searching: depth + pruning_ratios + quant_bits + exit_threshold

Generation 1/5
  [1/8] depth=16 exit_thr=0.5  acc=46.3%  lat=8.9ms  size=54.3MB  (26s)
  [2/8] depth=9 exit_thr=0.7  acc=46.2%  lat=5.4ms  size=11.6MB  (15s)
  [3/8] depth=12 exit_thr=0.3  acc=45.9%  lat=1.9ms  size=35.0MB  (19s)
  [4/8] depth=6 exit_thr=0.5  acc=49.6%  lat=3.7ms  size=20.5MB  (10s)
  [5/8] depth=9 exit_thr=0.5  acc=48.9%  lat=5.3ms  size=19.6MB  (15s)
  [6/8] depth=9 exit_thr=0.5  acc=46.7%  lat=5.3ms  size=30.6MB  (15s)
  [7/8] depth=12 exit_thr=0.9  acc=49.4%  lat=7.0ms  size=22.6MB  (20s)
  [8/8] depth=16 exit_thr=0.3  acc=45.3%  lat=1.9ms  size=47.7MB  (25s)
  Best → acc=49.6%  lat=3.7ms  size=20.5MB  fitness=0.4550  exit_thr=0.5  pareto=5

Generation 2/5
  [1/8] depth=

In [12]:
# ═══════════════════════════════════════════════════════════════════════════════
# Cell 14  →  REPLACE
# FIX: Tests early-exit on the *same* ViTSupernet (32×32 / 10 classes).
#      Was creating a separate AdaptiveViTSupernet at 224×224 which was
#      disconnected from the NAS-trained model.
# ═══════════════════════════════════════════════════════════════════════════════

import torch
from src.supernet.vit_supernet import ViTSupernet
from src.search_space.search_space import SearchSpace

print("=" * 70)
print("QUERY-ADAPTIVE MECHANISMS TEST  (CIFAR-10 supernet)")
print("=" * 70)

# Reload the trained supernet
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ViTSupernet(img_size=32, patch_size=4, num_classes=10,
                    max_depth=16, embed_dim=384, num_heads=6).to(device)
model.load_state_dict(torch.load("supernet_checkpoint.pt", map_location=device))
model.eval()
print("✓ Loaded trained supernet")

# CIFAR-10 sized test batch
x = torch.randn(4, 3, 32, 32).to(device)

print("\n--- Test 1: No early exit (full depth) ---")
with torch.no_grad():
    logits, meta = model(x, depth=12, exit_threshold=0.99)  # 0.99 → very unlikely to exit
print(f"  layers_used={meta['layers_used']}  early_exited={meta['early_exited']}")

print("\n--- Test 2: Aggressive early exit (low threshold) ---")
with torch.no_grad():
    logits, meta = model(x, depth=12, exit_threshold=0.1)
print(f"  layers_used={meta['layers_used']}  early_exited={meta['early_exited']}  "
      f"confidence={meta['confidence']:.4f}")

print("\n--- Test 3: Sweep thresholds and measure average compute used ---")
thresholds = [0.1, 0.3, 0.5, 0.7, 0.9]
for thr in thresholds:
    layers_total = 0
    exits = 0
    with torch.no_grad():
        for _ in range(20):
            _, meta = model(x, depth=12, exit_threshold=thr)
            layers_total += meta['layers_used']
            exits        += int(meta['early_exited'])
    avg_layers = layers_total / 20
    print(f"  threshold={thr:.1f}  avg_layers={avg_layers:.1f}/12  "
          f"exit_rate={exits/20*100:.0f}%  "
          f"compute_saved={(1 - avg_layers/12)*100:.0f}%")

print("\n--- Test 4: Compression + early exit combined ---")
search_space = SearchSpace("config/search_space.yaml")
config = search_space.random_sample()
depth = config.architecture.depth
pr    = config.compression.pruning.layer_wise_ratios[:depth]
qb    = config.compression.quantization.layer_wise_bits[:depth]
with torch.no_grad():
    logits, meta = model(x, depth=depth, pruning_ratios=pr, quant_bits=qb,
                         exit_threshold=0.5)
print(f"  depth={depth}  pruning={'on' if config.compression.pruning.enabled else 'off'}  "
      f"quant={'on' if config.compression.quantization.enabled else 'off'}")
print(f"  layers_used={meta['layers_used']}  early_exited={meta['early_exited']}")

print(f"\n✅ Query-Adaptive Mechanisms Test PASSED!")
print("=" * 70)

QUERY-ADAPTIVE MECHANISMS TEST  (CIFAR-10 supernet)
✓ Loaded trained supernet

--- Test 1: No early exit (full depth) ---
  layers_used=12  early_exited=False

--- Test 2: Aggressive early exit (low threshold) ---
  layers_used=3  early_exited=True  confidence=0.3702

--- Test 3: Sweep thresholds and measure average compute used ---
  threshold=0.1  avg_layers=3.0/12  exit_rate=100%  compute_saved=75%
  threshold=0.3  avg_layers=3.0/12  exit_rate=100%  compute_saved=75%
  threshold=0.5  avg_layers=12.0/12  exit_rate=0%  compute_saved=0%
  threshold=0.7  avg_layers=12.0/12  exit_rate=0%  compute_saved=0%
  threshold=0.9  avg_layers=12.0/12  exit_rate=0%  compute_saved=0%

--- Test 4: Compression + early exit combined ---
[2026-04-27 20:22:18] [src.search_space.search_space] [INFO] SearchSpace initialized from config/search_space.yaml
  depth=12  pruning=on  quant=off
  layers_used=12  early_exited=False

✅ Query-Adaptive Mechanisms Test PASSED!


In [13]:
# ═══════════════════════════════════════════════════════════════════════════════
# Cell 15  →  REPLACE
# FIX: Was testing on a fresh AdaptiveViTSupernet with random weights and
#      224×224 images. Now tests the same trained CIFAR-10 supernet.
#      Results now reflect what the NAS actually found.
# ═══════════════════════════════════════════════════════════════════════════════

import torch
import numpy as np

print("=" * 70)
print("ADAPTIVITY IMPACT ANALYSIS  (trained supernet)")
print("=" * 70)

# Use the `model` already loaded in Cell 14 (trained ViTSupernet)
model.eval()

thresholds = [0.1, 0.2, 0.3, 0.5, 0.7, 0.9]
layers_used_list = []
exit_rates       = []

# Use 50 CIFAR-10 val batches for analysis
val_iter = iter(proxy_val_loader)   # defined in Cell 7A

print("\nSweeping exit thresholds on real CIFAR-10 val data...")
for threshold in thresholds:
    layers_total = 0
    early_exits  = 0
    n_batches    = 0

    with torch.no_grad():
        for imgs, _ in proxy_val_loader:
            imgs = imgs.to(device)
            _, meta = model(imgs, depth=12, exit_threshold=threshold)
            layers_total += meta['layers_used']
            early_exits  += int(meta['early_exited'])
            n_batches    += 1
            if n_batches >= 10:   # 10 batches is enough for a stable estimate
                break

    avg_layers = layers_total / n_batches
    exit_rate  = (early_exits / n_batches) * 100

    layers_used_list.append(avg_layers)
    exit_rates.append(exit_rate)

    print(f"  threshold={threshold:.1f}  avg_layers={avg_layers:.1f}/12  "
          f"exit_rate={exit_rate:.0f}%  "
          f"compute_saved={(1 - avg_layers/12)*100:.0f}%")

print("\n" + "=" * 70)
print("SUMMARY")
print("=" * 70)
print(f"\nMin compute (thr=0.1): {(1 - layers_used_list[0]/12)*100:.0f}% saved")
print(f"Max compute (thr=0.9): {(1 - layers_used_list[-1]/12)*100:.0f}% saved")
print(f"\nThe NAS-found exit_threshold balances accuracy vs savings.")
print(f"Best config exit_threshold: "
      f"{getattr(results['best_config'], 'exit_threshold', 'N/A')}")
print("=" * 70)

ADAPTIVITY IMPACT ANALYSIS  (trained supernet)

Sweeping exit thresholds on real CIFAR-10 val data...
  threshold=0.1  avg_layers=3.0/12  exit_rate=100%  compute_saved=75%
  threshold=0.2  avg_layers=3.0/12  exit_rate=100%  compute_saved=75%
  threshold=0.3  avg_layers=3.0/12  exit_rate=100%  compute_saved=75%
  threshold=0.5  avg_layers=12.0/12  exit_rate=0%  compute_saved=0%
  threshold=0.7  avg_layers=12.0/12  exit_rate=0%  compute_saved=0%
  threshold=0.9  avg_layers=12.0/12  exit_rate=0%  compute_saved=0%

SUMMARY

Min compute (thr=0.1): 75% saved
Max compute (thr=0.9): 0% saved

The NAS-found exit_threshold balances accuracy vs savings.
Best config exit_threshold: 0.5


In [14]:
# Cell 19: Demonstrate the difference
print("="*70)
print("MOCK vs REAL EVALUATION")
print("="*70)

print("\nMOCK EVALUATION (Before):")
print("  accuracy = random.uniform(0.75, 0.95)")
print("  latency = 50 + depth * 5")
print("  Result: NAS optimizes NOISE ❌")

print("\nREAL EVALUATION (Now):")
print("  1. Train model 1 epoch on CIFAR-10")
print("  2. Measure accuracy on validation set")
print("  3. Measure latency with timer")
print("  Result: NAS optimizes REAL METRICS ✅")

print("\n" + "="*70)
print("WHY THIS MATTERS")
print("="*70)

print("\nScenario 1: Mock Evaluation")
print("  Config A: depth=6,  acc=0.91 (random), latency=80ms (formula)")
print("  Config B: depth=12, acc=0.79 (random), latency=140ms (formula)")
print("  NAS picks: A (because random rolled 0.91 > 0.79)")
print("  Reality: A might actually have 0.65 accuracy!")

print("\nScenario 2: Real Evaluation")
print("  Config A: depth=6,  acc=0.65 (MEASURED), latency=82ms (MEASURED)")
print("  Config B: depth=12, acc=0.78 (MEASURED), latency=145ms (MEASURED)")
print("  NAS picks: B (because REAL accuracy 0.78 > 0.65)")
print("  Reality: B is actually better ✓")

print("\n" + "="*70)
print("NEXT STEPS")
print("="*70)

print("\n1. Scale up evaluation:")
print("   - Increase dataset_size from 2000 to 10000")
print("   - Evaluate for 2-3 epochs instead of 1")
print("   - Run longer NAS search (10+ generations)")

print("\n2. Use better proxy task:")
print("   - ImageNet-21k subset (faster than full ImageNet)")
print("   - Different tasks (detection, segmentation)")

print("\n3. Integrate with hardware profiling:")
print("   - Measure latency on ACTUAL edge devices")
print("   - Not just CPU timing, but RPi/Jetson/Mobile")

print("\n4. Add compression-aware training:")
print("   - Pruning-aware fine-tuning")
print("   - Quantization-aware training")
print("   - Knowledge distillation from teacher")

print("="*70)

MOCK vs REAL EVALUATION

MOCK EVALUATION (Before):
  accuracy = random.uniform(0.75, 0.95)
  latency = 50 + depth * 5
  Result: NAS optimizes NOISE ❌

REAL EVALUATION (Now):
  1. Train model 1 epoch on CIFAR-10
  2. Measure accuracy on validation set
  3. Measure latency with timer
  Result: NAS optimizes REAL METRICS ✅

WHY THIS MATTERS

Scenario 1: Mock Evaluation
  Config A: depth=6,  acc=0.91 (random), latency=80ms (formula)
  Config B: depth=12, acc=0.79 (random), latency=140ms (formula)
  NAS picks: A (because random rolled 0.91 > 0.79)
  Reality: A might actually have 0.65 accuracy!

Scenario 2: Real Evaluation
  Config A: depth=6,  acc=0.65 (MEASURED), latency=82ms (MEASURED)
  Config B: depth=12, acc=0.78 (MEASURED), latency=145ms (MEASURED)
  NAS picks: B (because REAL accuracy 0.78 > 0.65)
  Reality: B is actually better ✓

NEXT STEPS

1. Scale up evaluation:
   - Increase dataset_size from 2000 to 10000
   - Evaluate for 2-3 epochs instead of 1
   - Run longer NAS search (1